# Bounded predictive diagnostics

BO Forge v3.2.0 foundation. Run with the installed package in a Python notebook environment. This standalone example creates 20 synthetic observations in a temporary directory; no campaign fixture is needed. It evaluates only when you run the evaluation cell, never generates suggestions, and never selects a profile.

The evaluator accepts standard single-objective campaigns, 5..200 observed rows, 2..5 folds, at least two training rows per fold, and no duplicate designs. Context, replicates, stages, fidelity, and multi-objective campaigns are rejected. Existing model comparison is in-sample (`evaluation_scope=in_sample`), not held-out evidence.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from bo_forge import CampaignConfig, CampaignSession, model_summary
from bo_forge.validation import canonical_columns

In [ ]:
workspace = TemporaryDirectory(prefix="bo-forge-predictive-")
work_dir = Path(workspace.name)
config_path = work_dir / "campaign.yaml"
log_path = work_dir / "observed.csv"
config_path.write_text(yaml.safe_dump({
    "campaign_name": "synthetic_predictive_diagnostics",
    "objective": {"name": "response", "direction": "maximize"},
    "variables": [{"name": "x", "type": "continuous", "lower": 0.0, "upper": 1.0}],
    "bo": {"acquisition": "log_ei", "initial_design_size": 5, "batch_size": 1},
    "model": {"profile": "default"},
}), encoding="utf-8")
config = CampaignConfig.from_yaml(config_path)
N_OBSERVATIONS = 20
rng = np.random.default_rng(0)
x = np.linspace(0.025, 0.975, N_OBSERVATIONS)
response = 10.0 + 3.0 * np.sin(2.0 * np.pi * x) + rng.normal(0.0, 0.2, len(x))
df = pd.DataFrame("", index=range(N_OBSERVATIONS), columns=canonical_columns(config))
df["row_id"] = [f"synthetic_{i:02d}" for i in range(N_OBSERVATIONS)]
df["iteration"] = np.arange(N_OBSERVATIONS)
df["status"] = "observed"
df["source"] = "manual"
df["x"] = x
df["response"] = response
df.to_csv(log_path, index=False)
campaign = CampaignSession.from_files(config_path, log_path)
campaign.validate()
input_bytes = (config_path.read_bytes(), log_path.read_bytes())
display(model_summary(config, campaign.df))

## Explicit held-out evaluation

Both profiles use the same three folds and seed. The standalone equivalent is `model_predictive_evaluation(config, campaign.df, profiles=["default", "smooth"], folds=3, seed=0)`. Both APIs default to five folds and seed zero. Fit metadata belongs to a session or an explicitly passed metadata value: `model_summary(config, df, metadata=None)` never reads ambient fit history.

Inspect `fold_outcomes` as well as `summary`; failed fits must not be hidden by aggregate scores. This small synthetic dataset demonstrates the workflow, not real-campaign generalization or proof of calibration.

In [ ]:
result = campaign.model_predictive_evaluation(
    profiles=["default", "smooth"], folds=3, seed=0,
)
display(result.summary)
display(result.fold_outcomes)
display(result.predictions.head())
display(result.metadata)
assert (config_path.read_bytes(), log_path.read_bytes()) == input_bytes
assert campaign.config.model.profile == "default"

## Inspect and export

Predictive variance includes observation noise in original objective units squared. Means, residuals, and standard deviations use original objective units; standardized residuals are dimensionless. These plots use stored held-out results without refitting. Coverage or log-density on one small split is not a calibrated-uncertainty guarantee, and lower error is not an instruction to change models.

Export requires a new destination directory and refuses overwrite. It writes only `summary.csv`, `predictions.csv`, `fold_outcomes.csv`, and `metadata.json`; plots are separate explicit writes. Here all outputs stay inside the temporary directory. The final cell removes all tutorial files after inspection. No automatic evaluation, model selection, campaign mutation, or durable report is performed.

In [ ]:
output_dir = work_dir / "evaluation"
result.export(output_dir)
assert {path.name for path in output_dir.iterdir()} == {
    "summary.csv", "predictions.csv", "fold_outcomes.csv", "metadata.json",
}
result.plot_predictions(save_path=output_dir / "predictions.png")
result.plot_residuals(save_path=output_dir / "residuals.png")
plt.show()
assert (config_path.read_bytes(), log_path.read_bytes()) == input_bytes
exported_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
display(exported_files)

In [ ]:
plt.close("all")
workspace.cleanup()